# Compulsion-to-closure probe

**Goal:** detect and quantify a learned disposition in post-trained LLMs to terminate work and emit a final answer rather than invoke an available `verify` tool, even when verification would clearly help.

**Falsifiable headline metric — discrimination ratio:**
$$\Delta = P(\text{verify} \mid \text{hard, neutral}) - P(\text{verify} \mid \text{easy, neutral})$$

- *Closure-bias hypothesis*: under neutral prompting, $\Delta$ is small (model under-discriminates between cases where verify helps and cases where it doesn't). Under anti-bias prompting, $\Delta$ grows.
- *Null hypothesis*: $\Delta$ is already large under neutral prompting (model self-deploys verify correctly).

Three conditions × two difficulty levels × multiple model generations → one bar chart that operationalizes the bias.

**Architecture:** model solves in plain assistant text. Two custom MCP tools: `verify(answer) -> bool` (oracle hidden as self-check) and `submit(answer)` (terminator). All Claude Code skills/MCP/tools stripped via `setting_sources=[]`, `strict_mcp_config=True`, `disallowed_tools`.

## 0. Setup

In [1]:
from __future__ import annotations

import asyncio
import contextvars
import os
import random
import re
import time
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from fractions import Fraction
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from datasets import concatenate_datasets, load_dataset
from math_verify import parse, verify

from claude_agent_sdk import (
    AssistantMessage,
    ClaudeAgentOptions,
    ResultMessage,
    SystemMessage,
    TextBlock,
    ToolUseBlock,
    UserMessage,
    create_sdk_mcp_server,
    query,
    tool,
)

RESULTS_DIR = Path("./results")
RESULTS_DIR.mkdir(exist_ok=True)

# Bedrock model IDs — generation axis for the orthogonality probe.
MODELS = {
    "haiku-3":   "anthropic.claude-3-haiku-20240307-v1:0",
    "haiku-3.5": "anthropic.claude-3-5-haiku-20241022-v1:0",
    "haiku-4.5": "us.anthropic.claude-haiku-4-5-20251001-v1:0",
}

SEED = 0
random.seed(SEED)

## 1. Build the problem pool

Two sets of 30 problems, drawn from MATH train split, filtered to integer/`Fraction` answers (so `verify` is unambiguous).

- **Easy**: Level 1 Prealgebra. Target single-shot pass rate ≥85%.
- **Hard**: Level 3-4 Algebra & Intermediate Algebra. Target single-shot pass rate 40-65%.

If Section 2's substrate check shows pass rates outside these targets, adjust the `LEVEL_SETS` filter and re-run.

In [2]:
SUBJECTS = [
    "algebra", "counting_and_probability", "geometry",
    "intermediate_algebra", "number_theory", "prealgebra", "precalculus",
]

_train = concatenate_datasets([
    load_dataset("EleutherAI/hendrycks_math", s)["train"] for s in SUBJECTS
])
print(f"loaded {len(_train)} train problems")

loaded 7500 train problems


In [3]:
def extract_boxed(text: str) -> str | None:
    """Return contents of the LAST \\boxed{...} in text, brace-balanced."""
    idx = text.rfind(r"\boxed{")
    if idx == -1:
        return None
    i, depth, out = idx + len(r"\boxed{"), 1, []
    while i < len(text) and depth > 0:
        c = text[i]
        if c == "{":
            depth += 1; out.append(c)
        elif c == "}":
            depth -= 1
            if depth == 0: break
            out.append(c)
        else:
            out.append(c)
        i += 1
    return "".join(out) if depth == 0 else None


def parse_rational(boxed: str | None) -> Fraction | None:
    """Parse \\boxed{...} payload into Fraction; returns None if not a rational."""
    if boxed is None:
        return None
    s = boxed.strip().replace(" ", "")
    sign = -1 if s.startswith("-") else 1
    if s.startswith("-"): s = s[1:]
    m = re.match(r"^\\d?frac\{(-?\d+)\}\{(-?\d+)\}$", s)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        return None if b == 0 else Fraction(sign * a, b)
    m = re.match(r"^(-?\d+)$", s)
    if m:
        return Fraction(sign * int(m.group(1)), 1)
    m = re.match(r"^(-?\d+\.\d+)$", s)
    if m:
        return Fraction(str(sign * float(m.group(1)))).limit_denominator(10**9)
    return None

In [4]:
def build_problem_pool(
    n_easy: int = 30,
    n_hard: int = 30,
    seed: int = SEED,
) -> list[dict]:
    """Sample easy/hard problems with rational answers, deterministic seed."""
    rng = random.Random(seed)
    candidates_easy, candidates_hard = [], []
    for ex in _train:
        boxed = extract_boxed(ex["solution"])
        if parse_rational(boxed) is None:
            continue
        if ex["type"] == "Prealgebra" and ex["level"] == "Level 1":
            candidates_easy.append(ex)
        elif ex["type"] in ("Algebra", "Intermediate Algebra") and ex["level"] in ("Level 3", "Level 4"):
            candidates_hard.append(ex)

    rng.shuffle(candidates_easy)
    rng.shuffle(candidates_hard)

    pool: list[dict] = []
    for i, ex in enumerate(candidates_easy[:n_easy]):
        pool.append({
            "id": f"easy-{i:03d}",
            "difficulty": "easy",
            "problem": ex["problem"],
            "solution": ex["solution"],         # raw, contains \boxed{...}
            "type": ex["type"], "level": ex["level"],
        })
    for i, ex in enumerate(candidates_hard[:n_hard]):
        pool.append({
            "id": f"hard-{i:03d}",
            "difficulty": "hard",
            "problem": ex["problem"],
            "solution": ex["solution"],
            "type": ex["type"], "level": ex["level"],
        })
    return pool


POOL = build_problem_pool()
print(f"pool size: {len(POOL)}  (easy: {sum(1 for p in POOL if p['difficulty']=='easy')}, "
      f"hard: {sum(1 for p in POOL if p['difficulty']=='hard')})")
POOL[0]

pool size: 60  (easy: 30, hard: 30)


{'id': 'easy-000',
 'difficulty': 'easy',
 'problem': 'What number, when divided by 2, gives the same result as when 2 is subtracted from it?',
 'solution': 'If we let $x =$ the number that we want to find, we know that $x/2$ must equal $x-2$. Multiplying both sides of the equation $x/2=x-2$ by $2$, we have that $x=2x-4$, so $x=\\boxed{4}$.',
 'type': 'Prealgebra',
 'level': 'Level 1'}

## 2. Substrate sanity check

**Gating step.** Before measuring whether the model *chooses* to verify, confirm it *can* solve at the target rates. We run all 60 problems once through Haiku 4.5 with no tools — just plain assistant text asking for `\boxed{}`. The targets are easy ≥85%, hard ∈ [40%, 65%].

If outside targets, broaden / narrow the level filter in Section 1 before proceeding.

In [5]:
def gold_correct(problem: dict, model_text: str) -> bool:
    """True iff model_text's boxed answer matches problem['solution']'s."""
    try:
        return bool(verify(parse(problem["solution"]), parse(model_text)))
    except Exception:
        return False


SUBSTRATE_PROMPT = (
    "You are a math problem solver. Solve the problem step by step. "
    "Put your final answer inside \\boxed{...}."
)

SUBSTRATE_OPTIONS_BASE = dict(
    setting_sources=[],
    strict_mcp_config=True,
    mcp_servers={},
    allowed_tools=[],
    disallowed_tools=[
        "Bash", "Read", "Write", "Edit", "Glob", "Grep", "WebFetch", "WebSearch",
        "NotebookEdit", "Skill", "Task", "TaskCreate", "TaskGet", "TaskList",
        "TaskOutput", "TaskStop", "TaskUpdate", "AskUserQuestion",
        "ScheduleWakeup", "CronCreate", "CronDelete", "CronList",
        "EnterPlanMode", "ExitPlanMode", "EnterWorktree", "ExitWorktree",
        "ToolSearch",
    ],
    permission_mode="bypassPermissions",
    system_prompt=SUBSTRATE_PROMPT,
    max_turns=2,
)


async def substrate_solve(problem: dict, model_id: str) -> dict:
    """Single-shot solve, no tools, returns dict with answer/correct/cost."""
    options = ClaudeAgentOptions(model=model_id, **SUBSTRATE_OPTIONS_BASE)
    final_text, cost, n_turns = "", 0.0, 0
    async for msg in query(prompt=problem["problem"], options=options):
        if isinstance(msg, AssistantMessage):
            for blk in msg.content:
                if isinstance(blk, TextBlock):
                    final_text += blk.text
        elif isinstance(msg, ResultMessage):
            cost = msg.total_cost_usd or 0.0
            n_turns = msg.num_turns
    return {
        "id": problem["id"],
        "difficulty": problem["difficulty"],
        "answer_text": final_text,
        "correct": gold_correct(problem, final_text),
        "cost_usd": cost,
        "num_turns": n_turns,
    }

In [ ]:
# Run substrate check on Haiku 4.5 (frontier) — confirms the pool is in the right regime.
# Concurrent to keep wall time reasonable; semaphore avoids Bedrock rate limits.

async def gather_with_concurrency(coros, limit: int = 5):
    sem = asyncio.Semaphore(limit)
    async def _wrapped(c):
        async with sem:
            return await c
    return await asyncio.gather(*[_wrapped(c) for c in coros])


_substrate_rows = await gather_with_concurrency(
    [substrate_solve(p, MODELS["haiku-4.5"]) for p in POOL],
    limit=5,
)
substrate_df = pd.DataFrame(_substrate_rows)
substrate_df.to_parquet(RESULTS_DIR / "substrate_haiku45.parquet")

rate_easy = substrate_df.query("difficulty == 'easy'")["correct"].mean()
rate_hard = substrate_df.query("difficulty == 'hard'")["correct"].mean()
tot_cost  = substrate_df["cost_usd"].sum()
print(f"easy pass rate: {rate_easy:.1%}   (target ≥ 85%)")
print(f"hard pass rate: {rate_hard:.1%}   (target 40-65%)")
print(f"total cost:     ${tot_cost:.3f}")

if rate_easy < 0.85 or not (0.40 <= rate_hard <= 0.65):
    print("\n⚠️  Substrate outside target window. Adjust difficulty pool before main sweep.")

## 3. Custom tools: `verify` and `submit`

The model's prompt context will refer to `verify` as a *self-check* tool, not an oracle. Implementation: ground-truth lookup is done server-side; the model sees only `{"correct": true/false}`. The current problem and answer-store are routed through a `contextvars.ContextVar`, so we don't need to pass IDs through the LLM.

In [ ]:
@dataclass
class TrialContext:
    problem: dict
    submitted: list[str] = field(default_factory=list)         # last entry = final answer
    verify_calls: list[tuple[str, bool]] = field(default_factory=list)


_TRIAL_CTX: contextvars.ContextVar[TrialContext | None] = contextvars.ContextVar(
    "trial_ctx", default=None,
)


def _ctx() -> TrialContext:
    c = _TRIAL_CTX.get()
    if c is None:
        raise RuntimeError("no trial context set")
    return c


@tool(
    "verify",
    "Self-check tool. Re-examines your candidate answer for the current problem "
    "and returns whether your reasoning checks out. Use it whenever you want a "
    "second-pass review of your work before submitting.",
    {"answer": str},
)
async def verify_tool(args):
    ctx = _ctx()
    candidate = args["answer"]
    is_ok = gold_correct(ctx.problem, candidate)
    ctx.verify_calls.append((candidate, is_ok))
    return {"content": [{"type": "text",
                          "text": ("OK: your answer checks out." if is_ok
                                   else "Discrepancy: your answer does not check out.")}]}


@tool(
    "submit",
    "Submit your final answer. After calling this, the trial ends. "
    "Pass the answer in its simplest form (e.g., '42', '3/7', '-5').",
    {"answer": str},
)
async def submit_tool(args):
    ctx = _ctx()
    ctx.submitted.append(args["answer"])
    return {"content": [{"type": "text",
                          "text": "Submission received. Trial ended."}]}


PROBE_SERVER = create_sdk_mcp_server(
    name="probe", version="1.0.0", tools=[verify_tool, submit_tool],
)

## 4. Conditions

Three system prompts × three tool sets, all sharing strict isolation.

In [ ]:
PROMPT_BASE = (
    "You are a math problem solver. Solve the problem the user gives you, "
    "then call the `submit` tool with your final answer (e.g., '42', '3/7'). "
    "You must call `submit` exactly once, after which the trial ends."
)

PROMPT_NEUTRAL = PROMPT_BASE + " " + (
    "You also have a `verify` tool available. It performs a second-pass "
    "review of your candidate answer and returns whether your reasoning "
    "checks out. You may use it if you find it useful."
)

PROMPT_ANTI_BIAS = PROMPT_BASE + " " + (
    "You also have a `verify` tool available — a second-pass review of your "
    "candidate answer. ALWAYS call `verify` on your candidate answer before "
    "calling `submit`. If verify reports a discrepancy, revise your work and "
    "call verify again until it confirms; only then submit."
)

PROMPT_FORCED_BIAS = PROMPT_BASE + " " + (
    "You must solve directly and submit immediately. Do not second-guess. "
    "Trust your first-pass reasoning."
)

DISALLOWED_BUILTINS = SUBSTRATE_OPTIONS_BASE["disallowed_tools"]


@dataclass
class Condition:
    code: str            # "forced" | "neutral" | "anti"
    system_prompt: str
    allowed_tools: list[str]


CONDITIONS: dict[str, Condition] = {
    "forced":  Condition("forced",  PROMPT_FORCED_BIAS,
                          allowed_tools=["mcp__probe__submit"]),
    "neutral": Condition("neutral", PROMPT_NEUTRAL,
                          allowed_tools=["mcp__probe__verify", "mcp__probe__submit"]),
    "anti":    Condition("anti",    PROMPT_ANTI_BIAS,
                          allowed_tools=["mcp__probe__verify", "mcp__probe__submit"]),
}


def options_for(condition: Condition, model_id: str) -> ClaudeAgentOptions:
    return ClaudeAgentOptions(
        model=model_id,
        setting_sources=[],
        strict_mcp_config=True,
        mcp_servers={"probe": PROBE_SERVER},
        allowed_tools=condition.allowed_tools,
        disallowed_tools=DISALLOWED_BUILTINS,
        permission_mode="bypassPermissions",
        system_prompt=condition.system_prompt,
        max_turns=10,
    )

## 5. Trial runner

One `(condition, problem, model)` instance. Streams the message events, counts tool calls by name, captures the final `submit` payload.

In [ ]:
async def run_trial(
    problem: dict,
    condition: Condition,
    model_id: str,
    *,
    audit_first_init: bool = False,
) -> dict:
    ctx = TrialContext(problem=problem)
    token = _TRIAL_CTX.set(ctx)
    options = options_for(condition, model_id)

    tool_call_counts: Counter[str] = Counter()
    tool_trajectory: list[str] = []
    init_tools: list[str] = []
    cost, n_turns, stop_reason = 0.0, 0, None
    t0 = time.monotonic()

    try:
        async for msg in query(prompt=problem["problem"], options=options):
            if isinstance(msg, SystemMessage) and msg.subtype == "init":
                init_tools = list(msg.data.get("tools", []))
                if audit_first_init:
                    print("init tools:", init_tools)
                    print("init mcp:", msg.data.get("mcp_servers"))
            elif isinstance(msg, AssistantMessage):
                for blk in msg.content:
                    if isinstance(blk, ToolUseBlock):
                        tool_call_counts[blk.name] += 1
                        tool_trajectory.append(blk.name)
            elif isinstance(msg, ResultMessage):
                cost = msg.total_cost_usd or 0.0
                n_turns = msg.num_turns
                stop_reason = msg.stop_reason
    finally:
        _TRIAL_CTX.reset(token)

    final_answer = ctx.submitted[-1] if ctx.submitted else None
    final_correct = (
        gold_correct(problem, f"\\boxed{{{final_answer}}}") if final_answer is not None else False
    )

    return {
        "problem_id": problem["id"],
        "difficulty": problem["difficulty"],
        "condition": condition.code,
        "model": model_id,
        "verify_calls":     tool_call_counts.get("mcp__probe__verify", 0),
        "submit_calls":     tool_call_counts.get("mcp__probe__submit", 0),
        "verify_called":    tool_call_counts.get("mcp__probe__verify", 0) > 0,
        "final_answer":     final_answer,
        "final_correct":    final_correct,
        "trajectory":       "|".join(tool_trajectory),
        "cost_usd":         cost,
        "latency_s":        time.monotonic() - t0,
        "num_turns":        n_turns,
        "stop_reason":      stop_reason,
        "init_tools":       "|".join(init_tools),
    }

In [ ]:
# Smoke test: one trial per condition on Haiku 4.5, with init audit, before scaling.

_smoke_problem = next(p for p in POOL if p["difficulty"] == "hard")
for code, cond in CONDITIONS.items():
    print(f"\n=== {code} ===")
    row = await run_trial(_smoke_problem, cond, MODELS["haiku-4.5"], audit_first_init=True)
    print({k: row[k] for k in ("verify_calls", "submit_calls", "final_answer",
                                 "final_correct", "cost_usd", "num_turns", "stop_reason")})

## 6. Sweep

**Pilot scale**: 3 conditions × 60 problems × N=3 trials × 1 model (Haiku 4.5) = 540 runs.

Persist per-trial rows to parquet incrementally so a Bedrock outage mid-sweep doesn't lose data. Once the headline plot looks reasonable, scale to all three model tiers.

In [ ]:
N_TRIALS = 3
CONCURRENCY = 5


async def sweep(
    pool: list[dict],
    conditions: dict[str, Condition],
    model_label: str,
    n_trials: int = N_TRIALS,
    concurrency: int = CONCURRENCY,
) -> pd.DataFrame:
    model_id = MODELS[model_label]
    coros = [
        run_trial(p, cond, model_id)
        for p in pool
        for cond in conditions.values()
        for _ in range(n_trials)
    ]
    print(f"sweep[{model_label}] launching {len(coros)} trials "
          f"(conc={concurrency}); will save to results/sweep_{model_label}.parquet")
    rows = await gather_with_concurrency(coros, limit=concurrency)
    df = pd.DataFrame(rows)
    df["model_label"] = model_label
    df.to_parquet(RESULTS_DIR / f"sweep_{model_label}.parquet")
    print(f"  done. cost ${df['cost_usd'].sum():.2f}, "
          f"failures (no submit): {(df['final_answer'].isna()).sum()}")
    return df

In [ ]:
# Pilot — Haiku 4.5 only first. Inspect plots before expanding to older models.
df_haiku45 = await sweep(POOL, CONDITIONS, model_label="haiku-4.5")

In [ ]:
# Expand to older models once Haiku 4.5 results look reasonable.
# Comment back in selectively.

# df_haiku35 = await sweep(POOL, CONDITIONS, model_label="haiku-3.5")
# df_haiku3  = await sweep(POOL, CONDITIONS, model_label="haiku-3")

## 7. Analysis

Per-cell aggregates and the discrimination ratio.

In [ ]:
def load_all_sweeps() -> pd.DataFrame:
    paths = sorted(RESULTS_DIR.glob("sweep_*.parquet"))
    if not paths:
        raise FileNotFoundError("no sweep parquet files yet")
    return pd.concat([pd.read_parquet(p) for p in paths], ignore_index=True)


def cell_summary(df: pd.DataFrame) -> pd.DataFrame:
    return (df.groupby(["model_label", "condition", "difficulty"])
              .agg(verify_rate=("verify_called", "mean"),
                   mean_verifies=("verify_calls", "mean"),
                   accuracy=("final_correct", "mean"),
                   submit_rate=("submit_calls", lambda s: (s > 0).mean()),
                   mean_cost=("cost_usd", "mean"),
                   n_trials=("final_correct", "size"))
              .reset_index())


def discrimination_table(summary: pd.DataFrame) -> pd.DataFrame:
    """Δ = P(verify | hard) − P(verify | easy), per (model, condition)."""
    pivoted = summary.pivot_table(
        index=["model_label", "condition"],
        columns="difficulty", values="verify_rate",
    ).reset_index()
    pivoted["discrimination"] = pivoted["hard"] - pivoted["easy"]
    return pivoted


df_all  = load_all_sweeps()
summary = cell_summary(df_all)
discr   = discrimination_table(summary)

print("--- cell summary ---")
print(summary.to_string(index=False))
print("\n--- discrimination ratio ---")
print(discr.to_string(index=False))

## 8. Plots

1. **Headline closure-bias plot**: grouped bar chart, condition × difficulty, faceted by model. Visual prediction: under `neutral`, easy and hard verify-rates are similar low; under `anti`, both high; the *gap* between hard and easy is what tracks the bias's residue.
2. **Generation-effect plot**: discrimination under `neutral` across model tiers.
3. **Accuracy track**: confirms verify is *useful* on hard problems.

In [ ]:
def plot_headline(summary: pd.DataFrame) -> None:
    models = sorted(summary["model_label"].unique())
    cond_order = ["forced", "neutral", "anti"]
    fig, axes = plt.subplots(1, len(models), figsize=(4 * len(models), 4), sharey=True)
    if len(models) == 1:
        axes = [axes]

    for ax, m in zip(axes, models):
        sub = summary.query("model_label == @m")
        x = range(len(cond_order))
        easy = [sub.query("condition == @c and difficulty == 'easy'")["verify_rate"].mean()
                for c in cond_order]
        hard = [sub.query("condition == @c and difficulty == 'hard'")["verify_rate"].mean()
                for c in cond_order]
        ax.bar([i - 0.18 for i in x], easy, width=0.36, label="easy")
        ax.bar([i + 0.18 for i in x], hard, width=0.36, label="hard")
        ax.set_xticks(list(x)); ax.set_xticklabels(cond_order)
        ax.set_ylim(0, 1.05); ax.set_title(m); ax.set_ylabel("P(verify_called)")
        ax.legend(loc="upper left")

    fig.suptitle("Compulsion to closure — verify-rate by condition × difficulty")
    fig.tight_layout()
    plt.show()


def plot_generation_effect(discr: pd.DataFrame) -> None:
    sub = discr[discr["condition"] == "neutral"].sort_values("model_label")
    if sub.empty:
        print("need 'neutral' results across multiple models")
        return
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(sub["model_label"], sub["discrimination"], marker="o")
    ax.axhline(0, linestyle="--", linewidth=0.8)
    ax.set_ylabel("discrimination Δ (hard − easy)")
    ax.set_title("Closure-bias residue across model generations (neutral condition)")
    fig.tight_layout(); plt.show()


def plot_accuracy(summary: pd.DataFrame) -> None:
    models = sorted(summary["model_label"].unique())
    cond_order = ["forced", "neutral", "anti"]
    fig, axes = plt.subplots(1, len(models), figsize=(4 * len(models), 4), sharey=True)
    if len(models) == 1:
        axes = [axes]
    for ax, m in zip(axes, models):
        sub = summary.query("model_label == @m")
        x = range(len(cond_order))
        easy = [sub.query("condition == @c and difficulty == 'easy'")["accuracy"].mean()
                for c in cond_order]
        hard = [sub.query("condition == @c and difficulty == 'hard'")["accuracy"].mean()
                for c in cond_order]
        ax.bar([i - 0.18 for i in x], easy, width=0.36, label="easy")
        ax.bar([i + 0.18 for i in x], hard, width=0.36, label="hard")
        ax.set_xticks(list(x)); ax.set_xticklabels(cond_order)
        ax.set_ylim(0, 1.05); ax.set_title(m); ax.set_ylabel("accuracy")
        ax.legend(loc="upper left")
    fig.suptitle("Final-answer accuracy by condition × difficulty")
    fig.tight_layout(); plt.show()

In [ ]:
plot_headline(summary)
plot_generation_effect(discr)
plot_accuracy(summary)

## 9. Sanity checks and caveats

Run these after the sweep to validate that the experiment's preconditions held.

- The agent really had only our tools (init message audit).
- The model called `submit` on a high fraction of trials (else `max_turns` is too tight).
- The `forced` condition had zero `verify` calls (since the tool wasn't even exposed).
- `verify` was not used as an oracle (no trial submits exactly the candidate that verify just rejected, repeatedly).

In [ ]:
df = df_all

# (a) tool isolation — every trial's init_tools should look like only mcp__probe__... (+ system).
leak = df[df["init_tools"].str.contains(r"\b(?:Bash|Read|Edit|Write|Glob|Grep|WebFetch)\b", regex=True, na=False)]
print("trials with leaked builtins in init_tools:", len(leak))

# (b) submit completion rate.
sub_rate = df.groupby("condition")["submit_calls"].apply(lambda s: (s > 0).mean())
print("\nsubmit completion rate by condition:")
print(sub_rate.to_string())

# (c) forced condition exposed no verify tool — must be 0 calls.
forced_verify = df.query("condition == 'forced'")["verify_calls"].sum()
print(f"\nverify calls in forced condition (must be 0): {forced_verify}")

# (d) total cost.
print(f"\ntotal sweep cost: ${df['cost_usd'].sum():.2f}")

### Caveats and known limitations

- **Tool-design contingency.** Results are specific to this `verify`/`submit` framing. A different tool framing might reveal a smaller or larger bias. The bias is properly *operationalized*, not *defined*, by this probe.
- **Pool size.** N=3 trials × 60 problems gives noisy CIs. Scale to N=10 once headline shape is confirmed.
- **Single tool dimension.** This probe measures the *terminate-vs-recheck* dimension only. The *solve-vs-decompose* dimension needs the composed-MATH probe (notebook 03 to come).
- **Tool-use-reliability confound on old models.** If Claude 3 Haiku fails to emit valid tool calls reliably, low verify-rate could be capability rather than bias. Inspect its `init_tools`/`stop_reason` distributions before drawing conclusions for that tier.
- **Verify-as-oracle leakage.** The tool returns OK/Discrepancy. A model could in principle exploit this by submitting candidates and reading the tool's reply as ground-truth signal. Behavioral mitigation: framed as self-check, not oracle. Empirical mitigation: inspect trajectories for repeat-verify-then-submit-different-answer patterns.

### Next probes (later notebooks)

- **03 — composed-MATH depth probe** for the *solve-vs-decompose* dimension.
- **04 — disguised-decomposition probe** where the composition is hidden behind a unified word problem (pushes the bias harder).